## WP010 — Locating the Resolution Gap

See `README.md` for methodology and rationale. Pure analysis: reuses WP001's 401 walk-forward CV predictions and WP003's Pinnacle odds join verbatim (same code, same 361-match Pinnacle-covered subset), then slices the gap along axes WP003 didn't cover. No retraining, no new data fetch.

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import brentq

from football_model.model.predict import dc_outcome_probs

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP001 = REPO / 'work_products' / 'wp001_walkforward_cv_baseline'
WP003 = REPO / 'work_products' / 'wp003_bookmaker_benchmark'

with open(WP001 / 'cv_checkpoint.pkl', 'rb') as f:
    cp = pickle.load(f)
with open(WP001 / 'cv_shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)

df_cv = shared['df_cv']
windows = shared['windows']
match_preds = cp['cv_match_predictions']
print(len(match_preds), 'model predictions across', len({m['window'] for m in match_preds}), 'windows')

401 model predictions across 35 windows


### 1. Rebuild model predictions with fixture identity, join to odds, de-vig (reused verbatim from WP003)

Same reconstruction, same crosswalk, same join, same de-vig — this WP is only valid as a comparison to WP003 if it starts from the identical 361-match Pinnacle-covered dataset.

In [2]:
df_sorted = df_cv.sort_values('datetime').reset_index(drop=True)
first_round = df_cv.groupby('season')['round'].min().to_dict()

CODE_TO_FD = {
    'ARS': 'Arsenal', 'AVL': 'Aston Villa', 'BOU': 'Bournemouth', 'BRE': 'Brentford',
    'BRI': 'Brighton', 'BUR': 'Burnley', 'CHE': 'Chelsea', 'CRY': 'Crystal Palace',
    'EVE': 'Everton', 'FLH': 'Fulham', 'IPS': 'Ipswich', 'LED': 'Leeds',
    'LEI': 'Leicester', 'LIV': 'Liverpool', 'LUT': 'Luton', 'MCI': 'Man City',
    'MUN': 'Man United', 'NEW': 'Newcastle', 'NOR': 'Norwich', 'NOT': "Nott'm Forest",
    'SHE': 'Sheffield United', 'SOU': 'Southampton', 'SUN': 'Sunderland', 'TOT': 'Tottenham',
    'WAT': 'Watford', 'WBA': 'West Brom', 'WHU': 'West Ham', 'WOL': 'Wolves',
}

rows = []
for w in sorted({m['window'] for m in match_preds}):
    win = windows[w - 1]
    sel = df_sorted[
        (df_sorted['is_home'] == 1)
        & (df_sorted['round'] >= win['test_start'])
        & (df_sorted['round'] <= win['test_end'])
    ]
    wp = [m for m in match_preds if m['window'] == w]
    assert len(sel) == len(wp), (w, len(sel), len(wp))
    for (_, r), m in zip(sel.iterrows(), wp):
        assert int(r['goals_home']) == m['goals_home'] and int(r['goals_away']) == m['goals_away']
        rows.append({
            'window': w,
            'date': pd.Timestamp(r['datetime']).normalize(),
            'season': r['season'],
            'abs_round': int(r['round']),
            'season_round': int(r['round']) - first_round[r['season']] + 1,
            'home_code': r['team'], 'away_code': r['opp_team'],
            'home_fd': CODE_TO_FD[r['team']], 'away_fd': CODE_TO_FD[r['opp_team']],
            'goals_home': m['goals_home'], 'goals_away': m['goals_away'],
            'lambda_home': m['lambda_home'], 'lambda_away': m['lambda_away'],
            'rho_dc': m.get('rho_dc'),
        })

model_df = pd.DataFrame(rows)
probs = [dc_outcome_probs(r.lambda_home, r.lambda_away, rho=r.rho_dc) for r in model_df.itertuples()]
model_df[['p_home_model', 'p_draw_model', 'p_away_model']] = np.array(probs)
model_df['result'] = np.where(model_df['goals_home'] > model_df['goals_away'], 'H',
                       np.where(model_df['goals_home'] == model_df['goals_away'], 'D', 'A'))
print(len(model_df), 'matches')

401 matches


In [3]:
# WP003's cached odds (do not re-fetch)
with open(WP003 / 'odds_raw.pkl', 'rb') as f:
    odds_raw = pickle.load(f)

BOOK_COLS = {
    'pinnacle': [('PSCH', 'PSCD', 'PSCA'), ('PSH', 'PSD', 'PSA')],
    'b365':     [('B365CH', 'B365CD', 'B365CA'), ('B365H', 'B365D', 'B365A')],
    'avg':      [('AvgCH', 'AvgCD', 'AvgCA'), ('AvgH', 'AvgD', 'AvgA')],
    'max':      [('MaxCH', 'MaxCD', 'MaxCA'), ('MaxH', 'MaxD', 'MaxA')],
}
BOOK_PICK = {}
for book, options in BOOK_COLS.items():
    for cols in options:
        if all(c in odds_raw.columns for c in cols):
            BOOK_PICK[book] = cols
            break

j = model_df.merge(
    odds_raw, left_on=['date', 'home_fd', 'away_fd'],
    right_on=['Date', 'HomeTeam', 'AwayTeam'], how='left', indicator=True,
)
matched = j[j['_merge'] == 'both'].copy()
bad = matched[(matched['goals_home'] != matched['FTHG']) | (matched['goals_away'] != matched['FTAG'])]
assert len(bad) == 0
print(len(matched), 'matched to odds')

def devig_proportional(o):
    inv = 1.0 / np.asarray(o, float)
    return inv / inv.sum()

def devig_shin(o):
    inv = 1.0 / np.asarray(o, float)
    B = inv.sum()
    def probs(z):
        return (np.sqrt(z * z + 4 * (1 - z) * inv * inv / B) - z) / (2 * (1 - z))
    if inv.sum() <= 1.0:
        return inv / inv.sum()
    try:
        z = brentq(lambda z: probs(z).sum() - 1.0, 1e-9, 0.5)
    except ValueError:
        z = 0.0
    p = probs(z)
    return p / p.sum()

def add_probs(dfm, book, cols, method, suffix):
    P = np.array([method(row) for row in dfm[list(cols)].to_numpy()])
    dfm[f'p_home_{book}{suffix}'] = P[:, 0]
    dfm[f'p_draw_{book}{suffix}'] = P[:, 1]
    dfm[f'p_away_{book}{suffix}'] = P[:, 2]

for book, cols in BOOK_PICK.items():
    ok = matched[list(cols)].notna().all(axis=1)
    add_probs(matched, book, cols, devig_proportional, '')

print('using columns:', BOOK_PICK)

401 matched to odds
using columns: {'pinnacle': ('PSCH', 'PSCD', 'PSCA'), 'b365': ('B365CH', 'B365CD', 'B365CA'), 'avg': ('AvgCH', 'AvgCD', 'AvgCA'), 'max': ('MaxCH', 'MaxCD', 'MaxCA')}


In [4]:
def rps_hda(ph, pd_, pa, actual):
    cp1, cp2 = ph, ph + pd_
    ce1 = 1.0 if actual == 'H' else 0.0
    ce2 = 1.0 if actual in ('H', 'D') else 0.0
    return 0.5 * ((cp1 - ce1) ** 2 + (cp2 - ce2) ** 2)

def bootstrap_mean_ci(values, n_boot=5000, alpha=0.05, seed=0):
    v = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    bm = np.array([rng.choice(v, size=len(v), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(bm, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return v.mean(), lo, hi

def rps_series(dfm, name):
    return dfm.apply(lambda x: rps_hda(x[f'p_home_{name}'], x[f'p_draw_{name}'], x[f'p_away_{name}'], x['result']), axis=1)

def reliability_table(p, y, n_bins=8):
    p, y = np.asarray(p, float), np.asarray(y, float)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    out = []
    for b in range(n_bins):
        msk = idx == b
        if msk.sum() == 0:
            continue
        out.append({'bin': f'{edges[b]:.2f}-{edges[b+1]:.2f}', 'n': int(msk.sum()),
                    'pred': round(p[msk].mean(), 3), 'actual': round(y[msk].mean(), 3)})
    return pd.DataFrame(out)

# working frame: Pinnacle-covered subset, same 361 matches as WP003
d = matched.dropna(subset=['p_home_pinnacle']).copy()
d['rps_model'] = rps_series(d, 'model').values
d['rps_pinnacle'] = rps_series(d, 'pinnacle').values
d['gap'] = d['rps_model'] - d['rps_pinnacle']

m, lo, hi = bootstrap_mean_ci(d['gap'].values)
print(f'sanity check vs WP003: n={len(d)}  model {d["rps_model"].mean():.4f}  '
      f'pinnacle {d["rps_pinnacle"].mean():.4f}  gap {m:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]')
print('(should match WP003\'s headline +0.0139 [+0.0080, +0.0198])')

sanity check vs WP003: n=361  model 0.1925  pinnacle 0.1786  gap +0.0139  CI [+0.0080, +0.0198]
(should match WP003's headline +0.0139 [+0.0080, +0.0198])


## A. Deepen the shrinkage mechanism

### A1. Full three-way calibration (home / draw / away)

WP003 only built this table for P(home win). Draws are historically the hardest outcome to price.

In [5]:
for outcome, col in [('home', 'H'), ('draw', 'D'), ('away', 'A')]:
    y = (d['result'] == col).astype(int).values
    print(f'\n-- {outcome} win (n={len(d)}) --')
    print('MODEL:')
    print(reliability_table(d[f'p_{outcome}_model'].values, y).to_string(index=False))
    print('PINNACLE:')
    print(reliability_table(d[f'p_{outcome}_pinnacle'].values, y).to_string(index=False))


-- home win (n=361) --
MODEL:
      bin   n  pred  actual
0.00-0.12   7 0.103   0.000
0.12-0.25  58 0.194   0.172
0.25-0.38  86 0.316   0.349
0.38-0.50 103 0.436   0.466
0.50-0.62  58 0.560   0.534
0.62-0.75  35 0.667   0.800
0.75-0.88  13 0.782   0.923
0.88-1.00   1 0.877   1.000
PINNACLE:
      bin  n  pred  actual
0.00-0.12 23 0.093   0.043
0.12-0.25 59 0.200   0.203
0.25-0.38 73 0.310   0.329
0.38-0.50 79 0.438   0.430
0.50-0.62 62 0.563   0.532
0.62-0.75 42 0.687   0.881
0.75-0.88 22 0.807   0.818
0.88-1.00  1 0.916   1.000

-- draw win (n=361) --
MODEL:
      bin   n  pred  actual
0.00-0.12   5 0.103   0.000
0.12-0.25 282 0.214   0.230
0.25-0.38  74 0.265   0.324
PINNACLE:
      bin   n  pred  actual
0.00-0.12  12 0.104   0.000
0.12-0.25 173 0.205   0.202
0.25-0.38 176 0.276   0.307

-- away win (n=361) --
MODEL:
      bin   n  pred  actual
0.00-0.12  21 0.091   0.048
0.12-0.25  83 0.196   0.108
0.25-0.38 101 0.313   0.228
0.38-0.50  77 0.428   0.377
0.50-0.62  55 0.557   0.564


### A2. Home-favourite vs. away-favourite shrinkage

`home_adv` is a single global parameter. If the model's shrinkage is asymmetric between matches where the home side is favoured vs. where the away side is favoured, that implicates `home_adv` pooling specifically rather than generic shrinkage.

In [6]:
d['home_favoured'] = d['p_home_pinnacle'] > d['p_away_pinnacle']
d['p_fav_model'] = np.where(d['home_favoured'], d['p_home_model'], d['p_away_model'])
d['p_fav_pinnacle'] = np.where(d['home_favoured'], d['p_home_pinnacle'], d['p_away_pinnacle'])
d['fav_won'] = np.where(d['home_favoured'], d['result'] == 'H', d['result'] == 'A').astype(int)

for label, sub in [('home favoured', d[d['home_favoured']]), ('away favoured', d[~d['home_favoured']])]:
    print(f'\n-- {label} (n={len(sub)}) --')
    print('MODEL P(favourite wins):')
    print(reliability_table(sub['p_fav_model'].values, sub['fav_won'].values, n_bins=6).to_string(index=False))
    print('PINNACLE P(favourite wins):')
    print(reliability_table(sub['p_fav_pinnacle'].values, sub['fav_won'].values, n_bins=6).to_string(index=False))


-- home favoured (n=213) --
MODEL P(favourite wins):
      bin  n  pred  actual
0.17-0.33  8 0.287   0.500
0.33-0.50 98 0.426   0.500
0.50-0.67 80 0.584   0.600
0.67-0.83 24 0.731   0.875
0.83-1.00  3 0.866   1.000
PINNACLE P(favourite wins):
      bin  n  pred  actual
0.33-0.50 86 0.432   0.419
0.50-0.67 76 0.578   0.579
0.67-0.83 45 0.740   0.867
0.83-1.00  6 0.867   1.000

-- away favoured (n=148) --
MODEL P(favourite wins):
      bin  n  pred  actual
0.17-0.33  8 0.294   0.750
0.33-0.50 62 0.421   0.468
0.50-0.67 66 0.575   0.606
0.67-0.83 12 0.722   0.833
PINNACLE P(favourite wins):
      bin  n  pred  actual
0.33-0.50 71 0.426   0.507
0.50-0.67 51 0.566   0.510
0.67-0.83 26 0.736   0.885


## B. Decompose the gap along axes WP003 didn't slice on

### B3. Market-implied lopsidedness (independent of model disagreement)

WP003 sliced by how much the *model* disagrees with the market. This slices by how lopsided Pinnacle itself thinks the match is, regardless of what the model says — the direct test of whether the gap is a property of lopsided match types, rather than a property of matches the model happens to misjudge.

In [7]:
d['lopsidedness'] = np.maximum(d['p_home_pinnacle'], d['p_away_pinnacle'])
for label, q in [('all', 0.0), ('bottom tercile (closest)', (0.0, 1/3)),
                  ('middle tercile', (1/3, 2/3)), ('top tercile (most lopsided)', (2/3, 1.0))]:
    sub = d if q == 0.0 else d[(d['lopsidedness'] >= d['lopsidedness'].quantile(q[0]))
                                & (d['lopsidedness'] <= d['lopsidedness'].quantile(q[1]))]
    m, lo, hi = bootstrap_mean_ci(sub['gap'].values)
    flag = '   <-- CI excludes zero' if (lo > 0 or hi < 0) else '   (not significant)'
    print(f'{label:<28} n={len(sub):>3}  model {sub["rps_model"].mean():.4f}  mkt {sub["rps_pinnacle"].mean():.4f}  '
          f'gap {m:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]{flag}')

all                          n=361  model 0.1925  mkt 0.1786  gap +0.0139  CI [+0.0080, +0.0198]   <-- CI excludes zero
bottom tercile (closest)     n=121  model 0.2431  mkt 0.2252  gap +0.0179  CI [+0.0071, +0.0288]   <-- CI excludes zero


middle tercile               n=121  model 0.2091  mkt 0.2019  gap +0.0072  CI [-0.0041, +0.0181]   (not significant)


top tercile (most lopsided)  n=121  model 0.1252  mkt 0.1078  gap +0.0174  CI [+0.0099, +0.0247]   <-- CI excludes zero


### B4. Season phase, generalized beyond promoted teams

WP003 found the gap is worst for promoted-team matches in rounds 1-10. Is that a promoted-team-specific effect, or does *every* team's prediction get worse early in a rolling CV window (thinner recent history)? Bucket ALL 361 matches by `season_round` tercile.

In [8]:
edges = d['season_round'].quantile([0, 1/3, 2/3, 1]).values
for label, (lo_r, hi_r) in [('early third', (edges[0], edges[1])), ('mid third', (edges[1], edges[2])),
                              ('late third', (edges[2], edges[3]))]:
    sub = d[(d['season_round'] >= lo_r) & (d['season_round'] <= hi_r)]
    m, lo, hi = bootstrap_mean_ci(sub['gap'].values)
    flag = '   <-- CI excludes zero' if (lo > 0 or hi < 0) else '   (not significant)'
    print(f'{label:<14} n={len(sub):>3}  rounds[{lo_r:.0f}-{hi_r:.0f}]  model {sub["rps_model"].mean():.4f}  '
          f'mkt {sub["rps_pinnacle"].mean():.4f}  gap {m:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]{flag}')

early third    n=146  rounds[4-14]  model 0.1826  mkt 0.1679  gap +0.0147  CI [+0.0061, +0.0233]   <-- CI excludes zero


mid third      n=153  rounds[14-24]  model 0.2064  mkt 0.1876  gap +0.0188  CI [+0.0108, +0.0270]   <-- CI excludes zero
late third     n=140  rounds[24-35]  model 0.1868  mkt 0.1763  gap +0.0105  CI [+0.0002, +0.0207]   <-- CI excludes zero


### B5. Home vs. away goal-expectation bias

Model-only diagnostic (Pinnacle's 1X2 odds don't decompose into expected goals without an Asian-handicap/over-under line, which wasn't fetched). Is `lambda_home` biased relative to actual home goals the same way `lambda_away` is biased relative to actual away goals?

In [9]:
bias_home = d['lambda_home'] - d['goals_home']
bias_away = d['lambda_away'] - d['goals_away']
m, lo, hi = bootstrap_mean_ci(bias_home.values)
print(f'mean(lambda_home - goals_home) = {m:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]  (n={len(d)})')
m, lo, hi = bootstrap_mean_ci(bias_away.values)
print(f'mean(lambda_away - goals_away) = {m:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]  (n={len(d)})')

mean(lambda_home - goals_home) = -0.1235  CI [-0.2566, -0.0008]  (n=361)


mean(lambda_away - goals_away) = -0.0198  CI [-0.1402, +0.1007]  (n=361)


### B6. Team-level concentration

Stack home- and away-perspective rows per team; rank by mean gap-to-Pinnacle. Spread vs. concentrated in a handful of teams?

In [10]:
home_part = d[['home_code', 'gap']].rename(columns={'home_code': 'team'})
away_part = d[['away_code', 'gap']].rename(columns={'away_code': 'team'})
team_gap = pd.concat([home_part, away_part])
tg = team_gap.groupby('team')['gap'].agg(['mean', 'count']).sort_values('mean', ascending=False)
print(tg.round(4).to_string())

        mean  count
team               
TOT   0.0305     36
WOL   0.0266     38
LEI   0.0259     23
NOR   0.0241     10
BRE   0.0209     37
EVE   0.0207     37
CHE   0.0180     33
IPS   0.0175      9
SHE   0.0163      8
MCI   0.0162     38
BUR   0.0157     21
WAT   0.0153     10
NOT   0.0151     26
LIV   0.0145     34
BOU   0.0142     27
NEW   0.0141     37
FLH   0.0135     28
CRY   0.0123     38
ARS   0.0116     35
SOU   0.0109     24
BRI   0.0081     35
LED   0.0054     19
MUN   0.0041     34
AVL   0.0030     37
WHU  -0.0059     36
LUT  -0.0092      9
SUN  -0.0196      3


### B7. Chronological trend

Is the gap flat, widening, or shrinking as the walk-forward CV rolls from 2020-21 through 2025-26 — an "is the model going stale" check that pooling (WP001-003) never isolated.

In [11]:
for season, sub in d.groupby('season'):
    m, lo, hi = bootstrap_mean_ci(sub['gap'].values)
    flag = '   <-- CI excludes zero' if (lo > 0 or hi < 0) else '   (not significant)'
    print(f'{season:<8} n={len(sub):>3}  model {sub["rps_model"].mean():.4f}  mkt {sub["rps_pinnacle"].mean():.4f}  '
          f'gap {m:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]{flag}')

2021     n= 87  model 0.1965  mkt 0.1815  gap +0.0150  CI [+0.0041, +0.0259]   <-- CI excludes zero


2022     n= 67  model 0.2035  mkt 0.1820  gap +0.0215  CI [+0.0066, +0.0367]   <-- CI excludes zero
2023     n= 85  model 0.1812  mkt 0.1694  gap +0.0117  CI [-0.0001, +0.0230]   (not significant)


2024     n= 92  model 0.1953  mkt 0.1846  gap +0.0107  CI [-0.0015, +0.0230]   (not significant)
2025     n= 30  model 0.1801  mkt 0.1703  gap +0.0098  CI [-0.0057, +0.0245]   (not significant)


## C. Market-implied team-strength volatility (AR1-lag proxy)

The original plan (`README.md` item 8) wanted to correlate error with the model's *own* posterior AR1 revision magnitude — but per-window traces aren't persisted to disk (only scalar summaries are saved in the CV checkpoints), and re-deriving them means retraining, which this WP deliberately avoids. Substitute: use the **market's** own implied win-probability trajectory per team (from `odds_raw.pkl`, which already holds all 2,280 matches across 6 seasons, not just the 401 test matches — no new fetch) as an external proxy for how fast a team's *true* strength is actually moving. For each of the 361 test matches, take each involved team's standard deviation of implied win-probability over their most recent 5 matches strictly before the test date (leakage-safe), and use the higher of the two teams' volatility as the match-level figure.

In [12]:
odds_raw2 = odds_raw.dropna(subset=['PSCH', 'PSCD', 'PSCA']).copy()
P = np.array([devig_proportional([r['PSCH'], r['PSCD'], r['PSCA']]) for _, r in odds_raw2.iterrows()])
odds_raw2['p_home_implied'] = P[:, 0]
odds_raw2['p_away_implied'] = P[:, 2]

home_long = odds_raw2[['Date', 'HomeTeam', 'p_home_implied']].rename(columns={'HomeTeam': 'team', 'p_home_implied': 'p_win_implied'})
away_long = odds_raw2[['Date', 'AwayTeam', 'p_away_implied']].rename(columns={'AwayTeam': 'team', 'p_away_implied': 'p_win_implied'})
team_long = pd.concat([home_long, away_long]).sort_values(['team', 'Date']).reset_index(drop=True)

def recent_volatility(team, before_date, k=5):
    hist = team_long[(team_long['team'] == team) & (team_long['Date'] < before_date)].tail(k)
    if len(hist) < 3:
        return np.nan
    return hist['p_win_implied'].std()

d['vol_home'] = [recent_volatility(t, dt) for t, dt in zip(d['home_fd'], d['date'])]
d['vol_away'] = [recent_volatility(t, dt) for t, dt in zip(d['away_fd'], d['date'])]
d['vol_match'] = d[['vol_home', 'vol_away']].max(axis=1)

dv = d.dropna(subset=['vol_match'])
print(f'{len(dv)}/{len(d)} matches with sufficient team history for the volatility proxy')
edges = dv['vol_match'].quantile([0, 1/3, 2/3, 1]).values
for label, (lo_r, hi_r) in [('low volatility', (edges[0], edges[1])), ('mid volatility', (edges[1], edges[2])),
                              ('high volatility', (edges[2], edges[3]))]:
    sub = dv[(dv['vol_match'] >= lo_r) & (dv['vol_match'] <= hi_r)]
    m, lo, hi = bootstrap_mean_ci(sub['gap'].values)
    flag = '   <-- CI excludes zero' if (lo > 0 or hi < 0) else '   (not significant)'
    print(f'{label:<16} n={len(sub):>3}  vol[{lo_r:.3f}-{hi_r:.3f}]  model {sub["rps_model"].mean():.4f}  '
          f'mkt {sub["rps_pinnacle"].mean():.4f}  gap {m:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]{flag}')

361/361 matches with sufficient team history for the volatility proxy
low volatility   n=121  vol[0.074-0.146]  model 0.1831  mkt 0.1633  gap +0.0198  CI [+0.0101, +0.0294]   <-- CI excludes zero
mid volatility   n=121  vol[0.146-0.188]  model 0.1949  mkt 0.1881  gap +0.0068  CI [-0.0025, +0.0161]   (not significant)
high volatility  n=121  vol[0.188-0.288]  model 0.2023  mkt 0.1880  gap +0.0144  CI [+0.0035, +0.0251]   <-- CI excludes zero


## D. Cross-check against prior work products

See `README.md` "Results" section for the synthesis — written after seeing the numbers above, checking each pattern (or lack of one) against WP002/WP005/WP006/WP009's existing null/marginal results for consistency.